# 02 - Baseline and model comparison

Owner: Nicolai. Interpretation: Philipp.

Goal: compare a majority-class baseline, logistic regression and random forest using seven inputs, then assess business implications.

**Current status:** sections 1-2 demonstrate preprocessing on one development-data fold. Sections 3-5 remain planned; no credit-risk classifier has been trained or evaluated. The 200 held-out test cases remain reserved for final evaluation.

## 1. Goal, assumptions, and experimental setup

The target is `bad_risk`: **0 = good risk; 1 = bad risk**. The fixed split contains 800 development cases and 200 test cases. Load only development data here.

Five stratified, shuffled folds (seed 42) will later compare models. Each round uses 640 training cases and 160 validation cases. Balanced accuracy is the provisional selection metric; bad-risk recall and error costs also matter. This cell selects the first fold only to demonstrate the data flow; it reports no predictive performance.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd()
if not (ROOT / "code" / "preprocessing.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "code" / "preprocessing.py").exists():
    raise FileNotFoundError("Run from the project directory or its notebooks directory.")
sys.path.insert(0, str(ROOT / "code"))

from preprocessing import NUMERIC_FEATURES, RANDOM_SEED, build_preprocessor, load_partition

X_dev, y_dev = load_partition()
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
training_rows, validation_rows = next(folds.split(X_dev, y_dev))
X_train, X_validation = X_dev.iloc[training_rows], X_dev.iloc[validation_rows]

print(f"Development: {len(X_dev)} cases with {X_dev.shape[1]} raw inputs")
print(f"Demonstration fold: {len(X_train)} training / {len(X_validation)} validation cases")
print("No test-partition analysis or classifier fitting in this demonstration.")

Development: 800 cases with 7 raw inputs
Demonstration fold: 640 training / 160 validation cases
No test-partition analysis or classifier fitting in this demonstration.


## 2. Data preprocessing

Shared factories live in `code/preprocessing.py`:

- **Five categorical inputs:** one-hot encoding creates a separate 0/1 column for each category observed in training. Codes such as A40 and A41 do not become a numerical ranking. Source categories such as A65 (unknown / no savings) remain valid categories.
- **Two numeric inputs:** standardize amount and duration for logistic regression using training means and standard deviations. Random forest keeps these numbers unscaled.
- **Unseen category:** `handle_unknown="ignore"` keeps the output shape stable and encodes that input's block as zeros. This does not teach the model the category's risk; a future demo should flag unsupported input categories.
- **Missing values:** the development-data review found none. No imputation, row deletion or feature removal is introduced.

In the example below, only the 640 training cases determine category columns and numeric scaling. Validation uses `transform`, never `fit`. More encoded columns do **not** mean more fields in the future seven-input demo.

In [2]:
# Logistic-regression variant: fit preprocessing on this training fold only.
preprocessor = build_preprocessor()
train_values = preprocessor.fit_transform(X_train)
scaler = preprocessor.named_transformers_["numeric"]
training_means = scaler.mean_.copy()
validation_values = preprocessor.transform(X_validation)

assert int(scaler.n_samples_seen_) == len(X_train)
np.testing.assert_allclose(training_means, X_train[NUMERIC_FEATURES].mean())
np.testing.assert_array_equal(scaler.mean_, training_means)
np.testing.assert_allclose(train_values[:, :2].mean(axis=0), 0, atol=1e-12)
assert train_values.shape[1] == validation_values.shape[1]

# Random-forest variant: encode categories, keep the two original numeric values.
tree_preprocessor = build_preprocessor(scale_numeric=False)
tree_train_values = tree_preprocessor.fit_transform(X_train)
tree_validation_values = tree_preprocessor.transform(X_validation)
np.testing.assert_array_equal(tree_train_values[:, :2], X_train[NUMERIC_FEATURES].to_numpy())
np.testing.assert_array_equal(tree_validation_values[:, :2], X_validation[NUMERIC_FEATURES].to_numpy())

print(f"Raw inputs: {X_train.shape[1]} -> encoded columns in this fold: {train_values.shape[1]}")
print(f"Training matrix: {train_values.shape}; validation matrix: {validation_values.shape}")
print(f"Scaler learned from {int(scaler.n_samples_seen_)} training cases only.")
display(pd.DataFrame({
    "input": NUMERIC_FEATURES,
    "training_mean": scaler.mean_,
    "training_standard_deviation": scaler.scale_,
}))
encoded_preview = pd.DataFrame(
    validation_values, columns=preprocessor.get_feature_names_out(), index=X_validation.index,
)
display(encoded_preview.head())

Raw inputs: 7 -> encoded columns in this fold: 31
Training matrix: (640, 31); validation matrix: (160, 31)
Scaler learned from 640 training cases only.


,input,training_mean,training_standard_deviation
0,credit_amount,3146.809375,2616.084874
1,duration_months,20.534375,11.712390


,numeric__credit_amount,numeric__duration_months,categorical__purpose_A40,categorical__purpose_A41,categorical__purpose_A410,categorical__purpose_A42,categorical__purpose_A43,categorical__purpose_A44,categorical__purpose_A45,categorical__purpose_A46,...,categorical__employment_duration_A71,categorical__employment_duration_A72,categorical__employment_duration_A73,categorical__employment_duration_A74,categorical__employment_duration_A75,categorical__credit_history_A30,categorical__credit_history_A31,categorical__credit_history_A32,categorical__credit_history_A33,categorical__credit_history_A34
row_id,,,,,,,,,,,,,,,,,,,,,
7,1.453007,1.320450,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.797830,0.808172,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
17,1.882657,0.808172,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
25,-0.677657,-1.240940,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
39,-1.027799,-0.984801,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


### How this connects to model training

This fitted preprocessor is a **single-fold teaching example**, not an artifact to reuse for the model comparison. Each validation round must learn its own transformations.

For the comparison, `build_model_pipeline(model, scale_numeric=True)` combines a fresh preprocessor and a cloned, unfitted classifier. Pass the **raw** development inputs into cross-validation; scikit-learn fits the entire pipeline separately inside each training fold. Use `scale_numeric=False` for random forest. Do not first transform all 800 development cases and then cross-validate those transformed values.

No fitted model is saved in this step. After model selection is frozen, the chosen complete pipeline will be refitted on all development cases and used for the final test and demo.

Implementation references: [scikit-learn preprocessing pitfalls](https://scikit-learn.org/stable/common_pitfalls.html), [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), and [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html).

## 3. Baseline

Train a simple reference approach and explain its results.

## 4. Additional models and comparison

Compare models using the same data split. Tune models using training/validation data; reserve test data for the final evaluation.

## 5. Results and business interpretation

Save metrics to `results/metrics.csv` and figures under `results/figures/`. Interpret benefits, error costs, risks, and limitations for the report.